# MEME FIMO Motif Scan

`run_meme_fimo_scan` uses MEME Suite [FIMO](https://meme-suite.org/meme/doc/fimo.html) (Find Individual Motif Occurrences) to scan sequences for occurrences of known motifs (position weight matrices) supplied in a [MEME](https://meme-suite.org/meme/) `.meme` file. It is backed by [`pymemesuite`](https://pypi.org/project/pymemesuite/), so no host MEME installation is required.

FIMO is described in Grant, Bailey & Noble (2011), ["FIMO: scanning for occurrences of a given motif"](https://doi.org/10.1093/bioinformatics/btr064), *Bioinformatics* 27(7):1017-1018.

In [1]:
from pathlib import Path

import proto_tools
from proto_tools.tools.gene_annotation.meme import (
    MEMEFimoScanInput,
    MEMEFimoScanConfig,
    run_meme_fimo_scan,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Display input docs
display_api_reference("meme-fimo-scan", "input", "run_meme_fimo_scan")

# Path to the bundled example MEME motif file (a 10 bp DNA motif, consensus GAGCTGGTCA)
_meme_dir = Path(proto_tools.__file__).resolve().parent / "tools" / "gene_annotation" / "meme" / "examples"
motif_path = _meme_dir / "example.meme"

# A DNA sequence containing two copies of the motif consensus (GAGCTGGTCA)
dna_seq = "GTTGAGCTGGTCAACAAGTTGAGCTGGTCAAC"

inputs = MEMEFimoScanInput(
    sequences=dna_seq,
    motifs=str(motif_path),
)

**Input** — `MEMEFimoScanInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Target sequences to scan, as a single string or a list of strings |
| `motifs` | `str \| pathlib.Path` | required | Path to a MEME-format (.meme) motif PWM file, e.g. from JASPAR. AssetRef supported |

In [3]:
# Display config docs
display_api_reference("meme-fimo-scan", "config", "run_meme_fimo_scan")

# Default config: p-value threshold 1e-4, scanning both strands | see docs above for all fields
config = MEMEFimoScanConfig()

**Config** — `MEMEFimoScanConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int \| None` | `3600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `threshold` | `float` | `0.0001` | Report matches with p-value <= this cutoff; lower is stricter (FIMO --thresh) |
| `both_strands` | `bool` | `True` | Scan both strands for nucleotide motifs; set False for single-strand. Ignored for protein |

In [4]:
# Display output docs
display_api_reference("meme-fimo-scan", "output", "run_meme_fimo_scan")

**Output** — `MEMEFimoScanOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.gene_annotation.meme.meme_fimo_scan.FimoSequenceMatches]` | `[]` | Per-sequence motif occurrences, aligned to the input sequences |

## Basic usage

Scan the DNA sequence against the example motif with the default configuration. The two consensus copies should each produce a match.

In [5]:
result = run_meme_fimo_scan(inputs, config)

print(f"Found {result.num_matches} motif occurrence(s)")

for match in result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}"
    )

Running run_meme_fimo_scan [00:00]

Found 2 motif occurrence(s)
  motif=MA0TEST, start=4, stop=13, strand=+, pvalue=7.68e-07
  motif=MA0TEST, start=21, stop=30, strand=+, pvalue=7.68e-07


## Advanced usage

Set `both_strands=False` to scan only the forward strand (FIMO `--norc`), and loosen `threshold` to report weaker matches.

In [6]:
advanced_config = MEMEFimoScanConfig(
    threshold=1e-3,
    both_strands=False,
)

advanced_result = run_meme_fimo_scan(inputs, advanced_config)

print(f"Found {advanced_result.num_matches} motif occurrence(s) on the forward strand")

for match in advanced_result.results[0].matches:
    print(
        f"  motif={match.motif_id}, start={match.start}, stop={match.stop}, "
        f"strand={match.strand}, pvalue={match.pvalue:.2e}, matched={match.matched_sequence}"
    )

Running run_meme_fimo_scan [00:00]

Found 2 motif occurrence(s) on the forward strand
  motif=MA0TEST, start=4, stop=13, strand=+, pvalue=7.68e-07, matched=GAGCTGGTCA
  motif=MA0TEST, start=21, stop=30, strand=+, pvalue=7.68e-07, matched=GAGCTGGTCA


## Export results

FIMO matches can be exported to CSV (one row per match) or JSON. CSV is the default format.

In [7]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="meme_fimo_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
